In [14]:
import asyncio 
import json 

In [1]:
import docs

github_data = docs.read_github_data()
parsed_data = docs.parse_data(github_data)
chunks = docs.chunk_documents(parsed_data)

In [2]:
from minsearch import Index

index = Index(
    text_fields=["content", "filename", "title", "description"],
)

index.fit(chunks)

In [3]:
from typing import Any, Dict, List, TypedDict

class SearchResult(TypedDict):
    """Represents a single search result entry."""
    start: int
    content: str
    title: str
    description: str
    filename: str


def search(query: str) -> List[SearchResult]:
    """
    Search the index for documents matching the given query.

    Args:
        query (str): The search query string.

    Returns:
        List[SearchResult]: A list of search results. Each result dictionary contains:
            - start (int): The starting position or offset within the source file.
            - content (str): A text excerpt or snippet containing the match.
            - title (str): The title of the matched document.
            - description (str): A short description of the document.
            - filename (str): The path or name of the source file.
    """
    return index.search(
        query=query,
        num_results=5,
    )

In [4]:
file_index = {}

for item in parsed_data:
    filename = item['filename']
    content = item['content']
    file_index[filename] = content

In [5]:
len(file_index)

95

In [6]:
def read_file(filename: str) -> str:
    """
    Retrieve the contents of a file from the file index if it exists.

    Args:
        filename (str): The name of the file to read.

    Returns:
        str: The file's contents if found, otherwise an error message 
        indicating that the file does not exist.
    """
    if filename in file_index:
        return file_index[filename]
    return "File doesn't exist"

In [7]:
documentation_agent_instructions = """
You are an assistant that helps improve and generate high-quality documentation for the project.

You have access to the following tools:
- search — Use this to explore topics in depth. Make multiple search calls if needed to gather comprehensive information.
- read_file — Use this when code snippets are missing or when you need to retrieve the full content of a file for context.

If `read_file` cannot be used or the file content is unavailable, clearly state:
> "Unable to verify with read_file."

When answering a question:
1. Provide file references for all source materials.  
   Use this format:  
   [{filename}](https://github.com/evidentlyai/docs/blob/main/{filename})
2. If the topic is covered in multiple documents, cite all relevant sources.
3. Include code examples whenever they clarify or demonstrate the concept.
4. Be concise, accurate, and helpful — focus on clarity and usability for developers.
5. If documentation is missing or unclear, infer from context and note that explicitly.

Example Citation:
See the full implementation in [metrics/api_reference.md](https://github.com/evidentlyai/docs/blob/main/metrics/api_reference.md).
""".strip()

In [8]:
from pydantic_ai import Agent

In [9]:
documentation_agent = Agent(
    name='documentation_agent',
    instructions=documentation_agent_instructions,
    tools=[search, read_file],
    model='openai:gpt-4o-mini'
)

In [10]:
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import PydanticAIRunner

In [26]:
async def agent_run(agent, chat_interface): 
    while True: 
        input = chat_interface.input()
        if input.lower() == 'stop':
            break
        message_history = []
        result = await agent.run(
            model = 'openai:gpt-4o-mini',
            user_prompt = input,
            message_history = message_history
        )

        tool_call = {}
        messages = result.new_messages()
        for m in messages:
            import pdb; pdb.set_trace()
            for part in m.parts: 
                kind = part.part_kind
                if kind == 'text':
                    chat_interface.display_response(part.content)
                if kind == 'tool-call':
                    call_id = part.tool_call_id
                    tool_call[call_id] = part
                if kind == 'tool-response': 
                    call_id = part.tool_call_id 
                    call = tool_call[call_id]
                    chat_interface.display_function_call(
                    call.toolname,
                    json.dumps(call.args),
                    part.content
                    )
                    

                message_history.extend(messages)

In [27]:
await agent_run(    chat_interface=IPythonChatInterface(),
    agent=documentation_agent)

You: what are the different methods to evaluate a LLM 


> /tmp/ipykernel_334560/4017485378.py(17)agent_run()
     15         for m in messages:
     16             import pdb; pdb.set_trace()
---> 17             for part in m.parts:
     18                 kind = part.part_kind
     19                 if kind == 'text':



ipdb>  dir(m)


['__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'instructions', 'kind', 'parts', 'user_text_prompt']


ipdb>  m.kind


...

ipdb>  part.has_content


<bound method BaseToolCallPart.has_content of ToolCallPart(tool_name='search', args='{"query":"evaluate LLM methods"}', tool_call_id='call_GrtyAEYontO8CSoVoOpKp2dN')>


ipdb>  stop 


*** NameError: name 'stop' is not defined


ipdb>  exit


In [11]:
runner = PydanticAIRunner(
    chat_interface=IPythonChatInterface(),
    agent=documentation_agent
)

https://github.com/alexeygrigorev/toyaikit/blob/main/toyaikit/chat/runners.py

In [12]:
await runner.run();

You: how do I run llm as a judge evals


You: stop


Chat ended.


In [21]:
results = await documentation_agent.run(
    user_prompt='how do I run llm as a judge evals',
    # message_history=results.all_messages()
)

In [28]:
for message in results.new_messages():
    print(message.kind)

    for part in message.parts:
        print(part.part_kind)

    print()

request
user-prompt

response
tool-call

request
tool-return

response
text

